In [328]:
### first batch ###
# [
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA1_first_328_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA2_second_100_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA3_third_100_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA4_forth_100_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA5_fifth_100_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA6_sisth_74_batch
# ]

### AA2_second_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA1_first_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA2_second_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA3_third_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA4_forth_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA5_fifth_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA6_sisth_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA7_seventh_100_batch

# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA1_first_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA2_second_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA3_third_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA4_forth_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA5_fifth_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA6_sisth_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA7_seventh_100_batch

# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA3_third_batch

# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA4_fourth_batch

In [ ]:
# {f_position}\{s_position}  AA2_second_batch\AA6_sisth_100_batch f"_pevidence.json", \AA3_third_batch\AA3_third_100_batch f"_p.json"
f_position = "AA3_third_batch"
s_position = "AA3_third_100_batch"

Move the files from `1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA1_first_100_batch` to the corresponding location: `D:\AA_project\AA_geo_app_regulation\dataset\1122apk\1122apk_privacy_policy_url\out_summary_json\AA2_second_batch\AA1_first_100_batch`.

Filter the files—for instance, by selecting URLs that contain specific keywords (such as "privacy")—and then remove any URLs that are obviously advertisements.

In [ ]:
import json
import re
from pathlib import Path
from urllib.parse import urlparse


# -------------------------
# Privacy policy keyword
# -------------------------

PRIVACY_KEYWORDS = [
    "privacy",
    "privacy-policy",
    "privacy_policy",
    "privacypolicy",
    "data-privacy",
    "data_privacy",
    "data-protection",
    "dataprotection",
    "personal-data",
    "gdpr",
]


# -------------------------
# Overt advertisement / Aggregated ad source
# -------------------------

AD_DOMAINS = {
    "aarki.com",
    "corp.aarki.com",

    "admixer.com",
    "admixplay.com",

    "bidmachine.io",
    "beeswax.com",

    "hybrid.ai",
    "hyperad.tech",
    "jampp.com",
    "kayzen.io",

    "persona.ly",
    "lifestreet.com",

    "discover-tech.io",
    "yeahmobi.com",
    "en.yeahmobi.com",

    "vlion.mobi",
    "mobgc.com",
    "wofhub.com",
}


def get_domain(url):
    try:
        domain = urlparse(url).netloc.lower()

        if domain.startswith("www."):
            domain = domain[4:]

        return domain

    except Exception:
        return ""


def contains_privacy_keyword(url):
    url_lower = url.lower()

    return any(
        keyword in url_lower
        for keyword in PRIVACY_KEYWORDS
    )


def is_ad_domain(url):
    domain = get_domain(url)

    for blocked_domain in AD_DOMAINS:

        if (
            domain == blocked_domain
            or domain.endswith("." + blocked_domain)
        ):
            return True

    return False


def is_from_ad_viewer(item):
    """
    If all hits for the URL come from paths like:

    assets/ad-viewer/adViewer.xxx.js

    ...then it can generally be assumed that this is an advertiser list built into the ad SDK,
    rather than the app's own privacy policy.
    """

    hits = item.get("hits", [])

    if not hits:
        return False

    for hit in hits:

        file_path = (
            hit.get("file", "")
            .lower()
            .replace("\\", "/")
        )

        if "assets/ad-viewer/" not in file_path:
            return False

    return True


def evaluate_candidate(item):
    """
    Evaluate a candidate URL.

    return：
        keep
        keep_reasons
        warnings
    """

    url = item.get("url", "").strip()

    keep_reasons = []
    warnings = []

    # -------------------------
    # 1. The URL must resemble a privacy policy.
    # -------------------------

    if not contains_privacy_keyword(url):
        return False, [], []


    keep_reasons.append(
        "privacy-related keyword in URL"
    )


    # -------------------------
    # 2. Overt advertisement / Aggregated ad source
    # -------------------------

    if is_ad_domain(url):
        return False, [], []


    # -------------------------
    # 3. ad-viewer built-in ad URL
    # -------------------------

    if is_from_ad_viewer(item):
        return False, [], []


    # -------------------------
    # 4. Some weak signals
    # -------------------------

    domain = get_domain(url)

    third_party_domains = {
        "google.com",
        "policies.google.com",
        "firebase.google.com",

        "adjust.com",
        "appsflyer.com",
        "applovin.com",
        "vungle.com",
        "onesignal.com",
        "unity3d.com",
    }

    for third_party in third_party_domains:

        if (
            domain == third_party
            or domain.endswith("." + third_party)
        ):
            warnings.append(
                "third-party privacy policy / SDK policy"
            )
            break

    return True, keep_reasons, warnings


def parse_filename(filename):
    """
    for example:

    com.brain.game.word.quiz-23802_urls.json

    ->

    apkname:
        com.brain.game.word.quiz

    version_code:
        23802
    """

    filename = Path(filename).name

    match = re.match(
        r"(.+)-([0-9]+)_urls\.json$",
        filename
    )

    if not match:
        raise ValueError(
            f"Unable to parse APK information from the file name.: {filename}"
        )

    apkname = match.group(1)
    version_code = match.group(2)

    return apkname, version_code


def process_file(input_file, output_file):

    input_file = Path(input_file)

    apkname, version_code = parse_filename(
        input_file.name
    )

    with open(
        input_file,
        "r",
        encoding="utf-8"
    ) as f:

        data = json.load(f)


    output_items = []

    original_urls = data.get("urls", [])


    for item in original_urls:

        keep, reasons, warnings = (
            evaluate_candidate(item)
        )

        if not keep:
            continue


        url = item.get("url", "")


        output_items.append({

            "apkname": apkname,

            "version_code": version_code,

            "url": url,

            "privacy_candidate_info": {

                "url": url,

                "keep_reasons": reasons,

                "warnings": warnings
            },

            "url_evidence_info": item
        })


    result = {
        apkname: output_items
    }


    # if output_file is None:

    #     output_file = (
    #         input_file.parent
    #         /
    #         f"{apkname}_{version_code}"
    #         f"_merged_privacy_url_evidence.json"
    #     )


    with open(
        output_file,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            result,
            f,
            ensure_ascii=False,
            indent=2
        )


    print(
        f"APK: {apkname}"
    )

    print(
        f"Version: {version_code}"
    )

    print(
        f"Number of original URLs: {len(original_urls)}"
    )

    print(
        f"Privacy candidates: {len(output_items)}"
    )

    print(
        f"Output file: {output_file}"
    )

In [331]:
# if __name__ == "__main__":

#     process_file(
#         "com.brain.game.word.quiz-23802_urls.json"
#     )

In [ ]:
source = Path(rf"1122apk\1122apk_privacy_policy_url\{f_position}\{s_position}")
target = Path(rf"1122apk\1122apk_privacy_policy_url\out_summary_json\{f_position}\{s_position}")

# source = Path(rf"1122apk\1122apk_privacy_policy_url\{f_position}")
# target = Path(rf"1122apk\1122apk_privacy_policy_url\out_summary_json\{f_position}")

# # File name examples: com.brain.game.word.quiz_23802_merged_privacy_url_evidence.json
# pat = re.compile(r"^(?P<apk>.+?)_(?P<num>\d+)_merged_privacy_url_evidence\.json$", re.IGNORECASE)

# # Collect all *_deduplicated source directories.
src_dirs = [p for p in source.glob("*") if p.is_file() and p.name.endswith("_urls.json")]
src_dirs

for src_file in src_dirs:


    apkname, version_code = parse_filename(
        src_file.name
    )

    output_file = (
        target
        /
        f"{apkname}_{version_code}"
        f"_p.json"
    )

    process_file(
        src_file, output_file
    )

    # print(f"Processing: {src_file} -> {output_file}")

In the directory `1122apk\1122apk_privacy_policy_url\out_summary_json\50apk`, there are numerous JSON files; the privacy policy (PP) URLs extracted from them contain excessive redundancy.
For instance, every JSON entry includes the `apkname` and `version`, yet these details need only appear once per file; similarly, the URL associated with each entry should appear only once rather than four times.
Some JSON files are completely empty; these can simply be moved to a new local directory.

For the results located in `D:\AA_project\AA_geo_app_regulation\dataset\1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA2_second_100_batch_deduplicated`: simply remove duplicate URL fields within the same JSON entry, and discard any entries that are empty.

Only one APK needs to be loaded at a time.

In [333]:
import json
import shutil
from pathlib import Path

In [334]:
### first batch ###
# [
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA1_first_328_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA2_second_100_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA3_third_100_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA4_forth_100_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA5_fifth_100_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA6_sisth_74_batch
# ]

### AA2_second_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA1_first_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA2_second_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA3_third_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA4_forth_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA5_fifth_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA6_sisth_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA7_seventh_100_batch

# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA1_first_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA2_second_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA3_third_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA4_forth_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA5_fifth_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA6_sisth_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA7_seventh_100_batch

In [ ]:
# ====== Path Configuration ======
str_p = rf"1122apk\1122apk_privacy_policy_url\out_summary_json\{f_position}\{s_position}"
# str_p = rf"1122apk\1122apk_privacy_policy_url\out_summary_json\{f_position}"
SRC_DIR = Path(rf"{str_p}")
OUT_DIR = Path(rf"{str_p}_deduplicated")
# EMPTY_DIR = Path(rf"{str_p}_empty")

OUT_DIR.mkdir(parents=True, exist_ok=True)
# EMPTY_DIR.mkdir(parents=True, exist_ok=True)
# SRC_DIR, OUT_DIR, EMPTY_DIR

In [ ]:
def dedup_list_of_dict(items):
    seen = set()
    out = []
    for x in items:
        if not isinstance(x, dict):
            continue
        key = json.dumps(x, ensure_ascii=False, sort_keys=True)
        if key not in seen:
            seen.add(key)
            out.append(x)
    return out


def clean_record(rec):
    apkname = rec.get("apkname")
    version_code = rec.get("version_code")
    url = (rec.get("url") or "").strip()  # Do not split; keep the entire string intact.

    pci = rec.get("privacy_candidate_info") or {}
    uei = rec.get("url_evidence_info") or {}

    keep_reasons = pci.get("keep_reasons", [])
    warnings = pci.get("warnings", [])

    raw_hits = uei.get("hits", []) if isinstance(uei.get("hits", []), list) else []
    hits = []
    for h in raw_hits:
        if isinstance(h, dict):
            h2 = dict(h)
            h2.pop("url", None)  # Remove the internal duplicate url
            hits.append(h2)
    hits = dedup_list_of_dict(hits)

    raw_usage = uei.get("usage", []) if isinstance(uei.get("usage", []), list) else []
    usage = dedup_list_of_dict([u for u in raw_usage if isinstance(u, dict)])

    out = {
        "apkname": apkname,
        "version_code": version_code,
        "url": url
    }

    # Only keep privacy_candidate_info if it's not empty
    pci_out = {}
    if isinstance(keep_reasons, list) and keep_reasons:
        pci_out["keep_reasons"] = keep_reasons
    if isinstance(warnings, list) and warnings:
        pci_out["warnings"] = warnings
    if pci_out:
        out["privacy_candidate_info"] = pci_out

    # Discard if `usage` is empty; also discard if `hits` is empty.
    uei_out = {}
    if hits:
        uei_out["hits"] = hits
    if usage:
        uei_out["usage"] = usage
    if uei_out:
        out["url_evidence_info"] = uei_out

    return out


def merge_record(base, new):
    """Merge records with the same top-level URL"""
    # Merge keep_reasons / warnings
    br = base["privacy_candidate_info"].get("keep_reasons", [])
    nr = new["privacy_candidate_info"].get("keep_reasons", [])
    bw = base["privacy_candidate_info"].get("warnings", [])
    nw = new["privacy_candidate_info"].get("warnings", [])

    base["privacy_candidate_info"]["keep_reasons"] = list(dict.fromkeys(br + nr))
    base["privacy_candidate_info"]["warnings"] = list(dict.fromkeys(bw + nw))

    # Merge hits / usage and deduplicate
    base_hits = base["url_evidence_info"].get("hits", [])
    new_hits = new["url_evidence_info"].get("hits", [])
    base_usage = base["url_evidence_info"].get("usage", [])
    new_usage = new["url_evidence_info"].get("usage", [])

    base["url_evidence_info"]["hits"] = dedup_list_of_dict(base_hits + new_hits)
    base["url_evidence_info"]["usage"] = dedup_list_of_dict(base_usage + new_usage)

    return base


summary = {"total": 0, "processed": 0, "moved_empty": 0, "failed": 0}

for fp in SRC_DIR.glob("*.json"):
    summary["total"] += 1
    try:
        txt = fp.read_text(encoding="utf-8", errors="ignore").strip()
        # if not txt:
        #     shutil.move(str(fp), str(EMPTY_DIR / fp.name))
        #     summary["moved_empty"] += 1
        #     continue

        try:
            obj = json.loads(txt)
        except json.JSONDecodeError:
            # shutil.move(str(fp), str(EMPTY_DIR / fp.name))
            # summary["moved_empty"] += 1
            continue

        # Compatible with structure：{apk_id: [records]}
        if isinstance(obj, dict) and len(obj) == 1 and isinstance(next(iter(obj.values())), list):
            apk_id = next(iter(obj.keys()))
            records = next(iter(obj.values()))
        else:
            # reveal all the details
            apk_id = fp.stem
            if isinstance(obj, list):
                records = obj
            elif isinstance(obj, dict):
                records = [obj]
            else:
                records = []

        cleaned_map = {}  

        for rec in records:
            if not isinstance(rec, dict):
                continue
            cleaned = clean_record(rec)
            url_key = cleaned["url"]
            if not url_key:
                continue

            if url_key not in cleaned_map:
                cleaned_map[url_key] = cleaned
            else:
                cleaned_map[url_key] = merge_record(cleaned_map[url_key], cleaned)

        out_records = list(cleaned_map.values())

        if not out_records:
            # shutil.move(str(fp), str(EMPTY_DIR / fp.name))
            # summary["moved_empty"] += 1
            continue

        out_obj = {apk_id: out_records}
        out_fp = OUT_DIR / fp.name
        out_fp.write_text(json.dumps(out_obj, ensure_ascii=False, indent=2), encoding="utf-8")

        summary["processed"] += 1

    except Exception as e:
        summary["failed"] += 1
        print(f"[FAILED] {fp.name}: {e}")

print("Processing complete:", summary)
print("Deduplicated output directory:", OUT_DIR)
# print("Empty file move directory:", EMPTY_DIR)

In [337]:
### first batch ###
# [
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA1_first_328_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA2_second_100_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA3_third_100_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA4_forth_100_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA5_fifth_100_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA6_sisth_74_batch
# ]

<!-- 指定目录为1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch，遍历所有以'_deduplicated'结尾的文件，然后遍历里面的文档，文件名是com.brain.game.word.quiz_23802_merged_privacy_url_evidence，其中com.brain.game.word.quiz是apkname，23802是编号，之后只根据文档名，一个apk只保存一个json文档，虽然编号不一样，但是内容是一样的，所以一个apk只需要保存一个json文档。将结果输出到新的目录吧。 -->